<a href="https://colab.research.google.com/github/IMMAN99/ml-nids/blob/main/notebooks/00_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nproc
!free -h
!df -h /content
!java -version

2
               total        used        free      shared  buff/cache   available
Mem:            12Gi       994Mi       9.3Gi       2.3Mi       2.7Gi        11Gi
Swap:             0B          0B          0B
Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   85M  108G   1% /
[0.024s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.024s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)


In [1]:
!pip install -q "pyspark==4.0.*" xgboost huggingface_hub
import pyspark; print(pyspark.__version__)

4.0.5


In [2]:
from google.colab import drive
drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/Big Data/Project/ML-NIDS"
!mkdir -p /content/data
!cp "{BASE}/dataset/"chunk_*.parquet /content/data/
!ls -lh /content/data
!du -sh /content/data

Mounted at /content/drive
total 16G
-rw------- 1 root root 2.9G Nov 27 15:03 chunk_01.parquet
-rw------- 1 root root 370M Nov 27 15:03 chunk_02.parquet
-rw------- 1 root root 311M Nov 27 15:03 chunk_03.parquet
-rw------- 1 root root 274K Nov 27 15:03 chunk_04.parquet
-rw------- 1 root root 2.8G Nov 27 15:04 chunk_05.parquet
-rw------- 1 root root 1.3G Nov 27 15:04 chunk_06.parquet
-rw------- 1 root root 269K Nov 27 15:04 chunk_07.parquet
-rw------- 1 root root 3.2G Nov 27 15:05 chunk_08.parquet
-rw------- 1 root root 1.4G Nov 27 15:06 chunk_09.parquet
-rw------- 1 root root 1.6G Nov 27 15:06 chunk_10.parquet
-rw------- 1 root root 288M Nov 27 15:06 chunk_11.parquet
-rw------- 1 root root 1.8G Nov 27 15:07 chunk_12.parquet
16G	/content/data


In [3]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .appName("ML-NIDS")
         .master("local[*]")
         .config("spark.driver.memory", "10g")
         .config("spark.sql.shuffle.partitions", "32")
         .getOrCreate())

df = spark.read.parquet("/content/data/*.parquet")
print("columns:", len(df.columns))
print("rows:", df.count())

columns: 323
rows: 48182782


In [4]:
labels = df.groupBy("label").count().orderBy(F.desc("count"))
labels.show(20, truncate=False)
labels.toPandas().to_csv(f"{BASE}/reports/label_counts.csv", index=False)
spark.stop()

+---------------+--------+
|label          |count   |
+---------------+--------+
|Benign         |43092133|
|DDoS_HOIC      |2164586 |
|DoS_HULK       |1824450 |
|DDoS_LOIC_HTTP |289527  |
|Bot            |286099  |
|Brute_Force_FTP|193360  |
|Brute_Force_SSH|188444  |
|DoS_SlowHTTP   |105550  |
|DoS_Golden_Eye |29485   |
|DoS_Slowloris  |8269    |
|Brute_Force_Web|536     |
|Brute_Force_XSS|232     |
|SQL_Injection  |106     |
|Infiltration   |5       |
+---------------+--------+

